<a href="https://colab.research.google.com/github/FarabiOnAMission/Machine-Learning-Stuffs/blob/main/Tensor_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from functools import reduce
import numpy as np

In [ ]:
#Although we know Tensor as some n dimensional data structure,
#but in reality in computer memory it is arranged as a flat 1D list of numbers,
#It accesses each data by doing a mathematical calculation known as strides

In [ ]:
#We pass data in Tensor by t = Tensor([1, 2, 3, 4, 5, 6], shape=(2, 3)) or T = Tensor([[1,2,3],[4,5,6]])
#So what we should do to make it operatable by computer is flatten the list and calculate the shape to make it usable by our stride function

In [ ]:
class Tensor:
  def __init__(self,data,shape=None):
    if isinstance(data,(list,tuple)):
      self._data, self._shape = self._flatten_nested(data)
    elif isinstance(data, np.ndarray):
      self._data = data.flatten().tolist() #flatten it and then convert it to a standard python list
      self._shape = tuple(data.shape)
    else :
      self.data = [data]
      self._shape = shape

    if shape is not None:
      total = reduce(lambda a, b: a*b, shape,1)
      if total != len(data):
        raise ValueError(f"Cannot reshape {len(self._data)} elements into shape {shape}")
      self._shape = tuple(shape)

    #Part for the Strides
    self._strides = self.calculate_strides(self._shape)

  def _flatten_nested(self, data):
        if len(data) == 0 or not isinstance(data[0], (list, tuple)):
            return list(data), (len(data),)
        flat = []
        shape = [len(data)]
        current = data
        while isinstance(current, (list, tuple)) and len(current) > 0:
            if isinstance(current[0], (list, tuple)):
                shape.append(len(current[0]))
            current = current[0]
        def _deflatten(item):
            for e in item:
                if isinstance(e, (list, tuple)): _deflatten(e)
                else: flat.append(e)
        _deflatten(data)
        return flat, tuple(shape)

  @staticmethod
  def calculate_strides(shape):
      if(len(shape)==0):
        return ()
      strides = [1] * len(shape)
      for i in range(len(shape) - 2, -1,-1):
        strides[i] = strides[i+1] * shape[i+1]
      return tuple(strides)

    #Part to reshape a tensor
  def reshape(self, new_shape):
      total_elements = len(self._data)
      shape_list = list(new_shape)

      if -1 in shape_list:
        known_product = 1
        for dim in shape_list:
          if dim != -1:
            known_product *= dim

        minus_one_index = shape_list.index(-1)
        shape_list[minus_one_index] = total_elements // known_product

        return Tensor(self._data, shape = tuple(shape_list))

  def __add__(self,other):
    if isinstance(other,(int,float)):
      new_data = [x + other for x in self._data]
      return Tensor(new_data, shape = self._shape)
    elif isinstance(other, Tensor):
      if self._shape != other._shape:
        raise ValueError("Cannot add tensors with different shapes")
      new_data = [x + y for x, y in zip(self._data, other._data)]
      return Tensor(new_data, shape = self._shape)
    else:
      raise TypeError("Unsupported operand type for +")

In [ ]:
t = Tensor(list(range(12)), shape=(2, 6))
r = t.reshape((3, 4))
r = t.reshape((-1, 3))